In [42]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import sys
sys.path.append("../")

import nest_asyncio
nest_asyncio.apply()

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "vscode"            

import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
import matplotlib.dates as mdates
plt.style.use('seaborn-v0_8-dark')
params = {'legend.fontsize': 'x-large',
        'figure.figsize': (16, 9),
        'axes.labelsize': 'x-large',
        'axes.titlesize':'x-large',
        'xtick.labelsize':'x-large',
        'ytick.labelsize':'x-large'}
pylab.rcParams.update(params)

import pandas as pd
import numpy as np
import QuantLib as ql
import rateslib as rl

import datetime
import pytz
NY_tz = pytz.timezone("America/New_York") 
CHI_tz = pytz.timezone("America/Chicago") 
UTC_tz = pytz.timezone("UTC")

import warnings
warnings.filterwarnings(
    "ignore",
    category=UserWarning,
)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [43]:
from MDP.IRSwaps.IRSwapsMDP import IRSwapsMDP
from SDRUtils.products.usd.usd_swaptions import USD_Swaptions
from SDRUtils.products._swaptions.pricer import (
    usd_swaption_straddle_pricer_from_row,
    usd_swaption_leg_pricer_from_row,
    usd_swaption_dealer_risk_reversal_skew_from_row,
    USDSwaptionStraddlePricerResult,
    USDSwaptionLegPricerResult,
	USDSwaptionDealerRiskReversalSkewResult,
    _compute_swaption_leg_greeks
)

In [44]:
cache_path = r"C:\Users\chris\clee\project-oasis\private\sdranalytics\.cache"

as_of = datetime.date(2026, 1, 15)
start = NY_tz.localize(datetime.datetime(as_of.year, as_of.month, as_of.day, 0, 0))
end = NY_tz.localize(datetime.datetime(as_of.year, as_of.month, as_of.day, 23, 59))

mdp = IRSwapsMDP(source="ERIS_EOD_LIVE-QL_BASIC")
pricer = mdp.get_pricer(request=dict(curve_name="USD-SOFR-1D", timestamp=start.date()))

from SDRUtils.data.builder import SDRDataBuilder
sdr = SDRDataBuilder(cache_path=cache_path, show_tqdm=True)
df = sdr.grab_sdr_trades(
	start_timestamp=start,
	end_timestamp=end,
	agency="CFTC",
	asset_class="RATES",
)
# df

MERGING SLICES...: 100%|██████████| 2/2 [00:00<00:00, 89.21it/s]


In [45]:
# sdf = USD_Swaptions().build_classification_dataframe(start=start, end=end, cache_path=cache_path)
sdf = USD_Swaptions().build_classification_dataframe(start=start, end=end, cache_path=cache_path, ignore_cache=True, merge_package_legs=True)
sdf

PRICING STRADDLES...: 100%|██████████| 226/226 [00:03<00:00, 63.29it/s]


,event_action,trade_id,execution_timestamp,effective_date,expiration_date,product_type,trade_label,notional,notional_currency,is_notional_capped,...,straddle_vega01,straddle_gamma01,straddle_theta1d,vega_curve_type,vega_curve_id,vega_curve_legs,vega_curve_vega01,vega_curve_weight,vega_curve_vega_ratio,vega_curve_pricing_method
0,MODI-TRAD,1732643977000000201,2026-01-15 05:13:03+00:00,2024-01-17 00:00:00,2026-01-20,SWAPTION_PAYER,USD-SOFR-COMPOUND 1D CONSTANT 2Yx1Y PAYER EURO...,120000000.0,USD,False,...,NaN,NaN,NaN,None,None,None,NaN,NaN,NaN,None
1,MODI-TRAD,1732644545000000101,2026-01-15 05:13:34+00:00,2024-01-17 00:00:00,2026-01-20,SWAPTION_PAYER,USD-SOFR-OIS Compound 1D CONSTANT 2Yx1Y PAYER ...,120000000.0,USD,False,...,NaN,NaN,NaN,None,None,None,NaN,NaN,NaN,None
2,MODI-TRAD,1733609362000000101,2026-01-15 07:21:02+00:00,2026-01-14 00:00:00,2026-05-19,SWAPTION_PAYER,USD-SOFR 1D CUSTOM 4Mx10Y PAYER EURO VANILLA CASH,250000000.0,USD,True,...,NaN,NaN,NaN,None,None,None,NaN,NaN,NaN,None
3,MODI-TRAD,1733609363000000201,2026-01-15 07:21:02+00:00,2026-01-14 00:00:00,2026-05-19,SWAPTION_RECEIVER,USD-SOFR 1D CUSTOM 4Mx10Y RECEIVER EURO VANILL...,250000000.0,USD,True,...,NaN,NaN,NaN,None,None,None,NaN,NaN,NaN,None
4,NEWT-NOVA,1743305405000000501,2026-01-15 10:31:26+00:00,2026-04-10 00:00:00,2026-04-10,SWAPTION_PAYER,USD-SOFR-OIS Compound 1D CONSTANT 3Mx10Y PAYER...,18000000.0,USD,False,...,NaN,NaN,NaN,None,None,None,NaN,NaN,NaN,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
319,NEWT-TRAD,1745369723000000101 / 1745369724000000201 / 17...,2026-01-15 19:31:22+00:00,2026-01-15 00:00:00,2026-07-15,SWAPTION_PAYER / SWAPTION_RECEIVER,USD-SOFR-OIS Compound 1D CONSTANT 6Mx1Y PAYER ...,500000000.0 / 200000000.0,USD,False,...,27539.005175,480.616987,-4307.966163,VEGA_DIAGONAL,VEGA_DIAGONAL_08644f1cdb15,"[1745369723000000101, 1745369724000000201, 174...",27539.005175174898 / 30242.377453203517,1.0,1.098165,QUANTLIB
320,NEWT-TRAD,1745385349000000401 / 1745385350000000501 / 17...,2026-01-15 19:34:01+00:00,2026-01-15 00:00:00,2026-07-15,SWAPTION_PAYER / SWAPTION_RECEIVER,USD-SOFR-OIS Compound 1D CONSTANT 6Mx1Y PAYER ...,250000000.0 / 70000000.0,USD,False,...,13769.502588,240.308493,-2153.983081,VEGA_TAIL_SPREAD,VEGA_TAIL_SPREAD_5733fb2fa5b7,"[1745385349000000401, 1745385350000000501, 174...",13769.502587587449 / 32439.000940767743,2.5,2.355859,QUANTLIB
321,NEWT-TRAD,1745415028000000301 / 1745415029000000401 / 17...,2026-01-15 19:38:27+00:00,2026-01-15 00:00:00,2027-01-15,SWAPTION_PAYER / SWAPTION_RECEIVER,USD-SOFR-OIS Compound 1D CONSTANT 1Yx10Y PAYER...,50000000.0 / 70000000.0,USD,False,...,32281.925725,336.276395,-3151.324945,VEGA_EXPIRY_SPREAD,VEGA_EXPIRY_SPREAD_de64803e247a,"[1745415028000000301, 1745415029000000401, 174...",32281.925724813762 / 32439.000940767743,1.0,1.004866,QUANTLIB
322,NEWT-TRAD,1745610739000000601 / 1745610741000000801 / 17...,2026-01-15T20:47:06+00:00 / 2026-01-15T20:47:1...,2026-01-15 00:00:00,2026-03-16,SWAPTION_RECEIVER / SWAPTION_PAYER,USD-SOFR-OIS Compound 1D CONSTANT IMM_H2026xIM...,1000000000.0 / 50000000.0,USD,False,...,31880.838454,978.752962,-11724.421803,VEGA_DIAGONAL,VEGA_DIAGONAL_f255dd820209,"[1745610741000000801, 1745610739000000601, 174...",31880.838453769902 / 89031.290176405,3.0,2.792627,QUANTLIB


In [55]:
# sdf[(sdf["package_type"] == "STRADDLE") & (sdf["trade_label"].str.contains("3Mx7Y"))]["trade_label"].to_list()

# sdf["package_type"].value_counts()

sdf[(sdf["trade_id"].str.contains("1744496707000000401"))]

# 1744193838000000501
# 1744496707000000401



# temp = sdf[(sdf["package_type"] == "STRADDLE") & (sdf["trade_label"].str.contains("1Yx10Y"))]
# temp
# ["platform_identifier"].value_counts()

# temp = sdf[sdf["package_type"].str.contains("SWAPTION")]
# temp["execution_timestamp"] = temp["execution_timestamp"].astype(str)
# temp.to_excel(f"{as_of.strftime("%Y-%m-%d")}_straddles.xlsx")
# temp
# ["trade_label"].value_counts()
# sdf[sdf["trade_label"].str.contains("1Yx10Y")]
# sdf.loc[361]
# ["trade_label"]

# sdf[sdf["package_type"].str.contains("RISK_R")]["platform_identifier"].value_counts()

# sdf[(sdf["package_type"].str.contains("RISK_R")) & (sdf["trade_label"].str.contains("2Yx10Y"))].to_dict(orient="records")
# temp["execution_timestamp"] = temp["execution_timestamp"].astype(str)
# temp.to_excel("2026-01-15_sdr_swaption.xlsx")

,event_action,trade_id,execution_timestamp,effective_date,expiration_date,product_type,trade_label,notional,notional_currency,is_notional_capped,...,straddle_vega01,straddle_gamma01,straddle_theta1d,vega_curve_type,vega_curve_id,vega_curve_legs,vega_curve_vega01,vega_curve_weight,vega_curve_vega_ratio,vega_curve_pricing_method
97,CORR-TRAD,1744496707000000401,2026-01-15 17:39:50+00:00,2026-01-15 00:00:00,2027-01-15,SWAPTION_PAYER,USD-SOFR-OIS Compound 1Y CONSTANT 1Yx10Y PAYER...,50000000.0,USD,False,...,NaN,NaN,NaN,None,None,None,NaN,NaN,NaN,None


In [17]:
import ujson as json


def format_swaption_pricing_results(
    results: USDSwaptionStraddlePricerResult | USDSwaptionLegPricerResult | USDSwaptionDealerRiskReversalSkewResult,
):
    if isinstance(results, USDSwaptionDealerRiskReversalSkewResult):
        output = {
            "trade": results.trade_label,
            "atm_strike": results.atm_strike * 100,
            "otm_payer_strike": results.otm_payer_strike * 100,
            "otm_receiver_strike": results.otm_receiver_strike * 100,
            "wing_strike_width": results.wing_strike_width,
            "atm_bpvol": results.atm_bpvol_yr,
            "otm_payer_bpvol": results.otm_payer_bpvol_yr,
            "otm_receiver_bpvol": results.otm_receiver_bpvol_yr,
            "payer_skew_bpvol_yr": results.payer_skew_bpvol_yr,
            "receiver_skew_bpvol_yr": results.receiver_skew_bpvol_yr,
            "skew_bpvol": results.skew_bpvol_yr,
            "atm_notional": results.atm_notional,
            "wing_notional": results.wing_notional,
            "otm_payer_vega01": results.otm_payer_vega01,
            "otm_receiver_vega01": results.otm_receiver_vega01,
            "dv01": results.dv01,
            "gamma01": results.gamma01,
            "vega01": results.vega01,
            "theta1d": results.theta1d,
            "wing_dv01": results.wing_dv01
        }
    else:
        output = {
            "trade": results.trade_label,
            "prem": (results.fwd_prem / results.notional) * 10_000,
            "bpvol": results.bpvol_yr,
            "bpvol_day": results.bpvol_yr / np.sqrt(252),
            "dv01": results.dv01,
            "gamma01": results.gamma01,
            "vega01": results.vega01,
            "theta1d": results.theta1d,
        }

    print(json.dumps(output, indent=4))

In [66]:
# format_swaption_pricing_results(usd_swaption_straddle_pricer_from_row(sdf.loc[199], pricer))
format_swaption_pricing_results(usd_swaption_leg_pricer_from_row(sdf.loc[96], pricer))

# format_swaption_pricing_results(usd_swaption_dealer_risk_reversal_skew_from_row(risk_reversal_row=sdf.loc[286], pricer=pricer))

{
    "trade": "USD-SOFR-OIS Compound 1Y CONSTANT 1Yx10Y RECEIVER EURO VANILLA PHYS",
    "prem": 237.0,
    "bpvol": 74.2427218072508,
    "bpvol_day": 4.676851870441368,
    "dv01": -19594.56338555966,
    "gamma01": 164.51342262480722,
    "vega01": 16134.515265982916,
    "theta1d": 1642.0426301111002
}


In [53]:

row = sdf.loc[168]

display(row.to_dict())
_compute_swaption_leg_greeks(
	pricer,
	row["expiration_date"],
	row["underlying_expiration_date"],
	row["strike"],
	row["notional"],
	row["premium"],
	"receiver" if "rec" in row["product_type"].lower() else "payer",
)

{'event_action': 'NEWT-TRAD',
 'trade_id': '1745734985000000401',
 'execution_timestamp': Timestamp('2026-01-15 20:56:16+0000', tz='UTC'),
 'effective_date': Timestamp('2026-01-15 00:00:00'),
 'expiration_date': Timestamp('2028-01-10 00:00:00'),
 'product_type': 'SWAPTION_PAYER',
 'trade_label': 'USD-SOFR-OIS Compound 1Y CONSTANT 2Yx30Y PAYER EURO VANILLA PHYS',
 'notional': 100000000.0,
 'notional_currency': 'USD',
 'is_notional_capped': False,
 'package_type': 'SWAPTION',
 'package_id': None,
 'package_legs': None,
 'underlying_expiration_date': Timestamp('2058-01-12 00:00:00'),
 'tenor_years': 30.027397260273972,
 'tenor_label': '30Y',
 'forward_start_years': 1.9863013698630136,
 'forward_label': '2Y',
 'premium': 450000.0,
 'exercise_style': 'EUROPEAN',
 'strike': 0.05234,
 'upi_underlier_name': 'NA/Swap OIS USD',
 'unique_product_identifier': 'QZZLNQ2D4JQT',
 'platform_identifier': 'BILT',
 'cleared': 'N',
 'package_indicator': False,
 'package_transaction_price': '',
 'option_pre

_SwaptionLegGreeks(bpvol_yr=50.80008511489668, dv01=13146.336552531715, gamma01=394.24023436883004, vega01=34351.294088425864, theta1d=1203.1005319558317, strike_offset=100)

In [452]:
# temp = sdf.loc[422].copy()

# temp["premium"] = temp["premium"] / 4
# format_swaption_pricing_results(usd_swaption_leg_pricer_from_row(temp, pricer))
# format_swaption_pricing_results(usd_swaption_straddle_pricer_from_row(temp, pricer))

In [123]:
# ids = [1712299926000000501, 1712299925000000401]

ids = [
1745547082000000201,
1745547081000000101


]

# df[df["Dissemination Identifier"].isin([str(id) for id in ids])].to_csv("_temp_raw_raw_trades.csv",index=False)

df[df["Dissemination Identifier"].isin([str(id) for id in ids])].to_dict(orient="records")
# df[df["Original Dissemination Identifier"].isin([str(id) for id in ids])].to_dict(orient="records")
# df["Action type"].value_counts()
# df["Event type"].value_counts()

[{'Dissemination Identifier': '1745547082000000201',
  'Original Dissemination Identifier': '',
  'Action type': 'NEWT',
  'Event type': 'TRAD',
  'Event timestamp': Timestamp('2026-01-15 19:23:44+0000', tz='UTC'),
  'Amendment indicator': None,
  'Asset Class': 'IR',
  'Product name': None,
  'Cleared': 'N',
  'Mandatory clearing indicator': False,
  'Execution Timestamp': Timestamp('2026-01-15 19:23:44+0000', tz='UTC'),
  'Effective Date': Timestamp('2026-01-15 00:00:00'),
  'Expiration Date': Timestamp('2026-02-17 00:00:00'),
  'Maturity date of the underlier': datetime.date(2056, 2, 19),
  'Non-standardized term indicator': False,
  'Platform identifier': 'BILT',
  'Prime brokerage transaction indicator': False,
  'Block trade election indicator': False,
  'Large notional off-facility swap election indicator': False,
  'Notional amount-Leg 1': '60,000,000',
  'Notional amount-Leg 2': '60,000,000',
  'Notional currency-Leg 1': 'USD',
  'Notional currency-Leg 2': 'USD',
  'Notional q